In [1]:
import requests
import re
import pandas as pd
import numpy as np
import math
import json
from itertools import chain
import time
from datetime import datetime
from IPython.display import clear_output

In [2]:
url = f'https://diavgeia.gov.gr/luminapi/api/search/export?q=q:"ελληνικο"&fq=thematicCategory:"περιβαλλον"&fq=organizationUid:99201077&fq=unitUid:77540&decisionTypeUid:2.4.6.1&fq=submissionTimestamp:[DT(2022-01-01T00:00:00) TO DT(2023-01-01T23:59:59)]&fq=issueDate:[DT(2022-01-01T00:00:00) TO DT(2023-01-01T23:59:59)]&wt=json'

In [ ]:
# 99201077 - τεχνικό επιμελητήριο
# decisionTypeUid:2.4.6.1 πράξεις χωροταξικού σχεδιασμού
from datetime import date
from dateutil.relativedelta import relativedelta

start_date = date(2019, 1, 1)
end_date = date(2026, 1, 25)
current_date = start_date
results=[]
while current_date < end_date:
    u = f'https://diavgeia.gov.gr/luminapi/api/search/export?q=q:"ελληνικο"&fq=thematicCategory:"περιβαλλον"&fq=organizationUid:99201077&fq=unitUid:77540&decisionTypeUid:2.4.6.1&fq=submissionTimestamp:[DT({current_date}T00:01:00) TO DT({current_date + relativedelta(months=+6)}T00:00:00)]&fq=issueDate:[DT({current_date}T00:01:00) TO DT({current_date + relativedelta(months=+6)}T00:00:00)]&wt=json'

    d = requests.get(u).json()
    results.extend(d['decisionResultList'])
    print(f"Getting date: {current_date}")
    current_date += relativedelta(months=+6)

Getting date: 2019-01-01
Getting date: 2019-07-01
Getting date: 2020-01-01
Getting date: 2020-07-01
Getting date: 2021-01-01
Getting date: 2021-07-01
Getting date: 2022-01-01
Getting date: 2022-07-01
Getting date: 2023-01-01
Getting date: 2023-07-01
Getting date: 2024-01-01
Getting date: 2024-07-01
Getting date: 2025-01-01
Getting date: 2025-07-01
Getting date: 2026-01-01


In [4]:
len(results)

11710

In [5]:
terms = ["Α-Π1", "Α-Π2", "Α-Π3", "Α-Π4", "Α-Π5", "Α-Π6", "ΠΜ-Π1", "Α-Α1-1", "ΠΜ Α1", "ΠΜ Α2"]

In [6]:
term_gen = (term for term in terms)

In [7]:
next(term_gen)

'Α-Π1'

In [8]:
filtered_results = []
for item in results:
    if "μητροπολιτικ" in item['subject'].lower() and "ελληνικ" in item['subject'].lower():
        print(item['subject'])
        filtered_results.append(item)

Έγκριση Εργασιών Δόμησης Μικρής Κλίμακας: Εκτέλεση Γεωτεχνικών και Γεωφυσικών ερευνών, Γεωτεχνική αξιολόγηση και Γεωτεχνικές μελέτες θεμελίωσης στο χερσαίο τμήμα του Μητροπολιτικού πόλου Ελληνικού – Αγ;iου Κοσμά
Οικοδομική Άδεια (ν.4759/2020): ΕΡΓΑΣΙΕΣ ΑΠΟΚΑΤΑΣΤΑΣΗΣ ΚΕΛΥΦΟΥΣ TOY ΧΑΡΑΚΤΗΡΙΣΜΕΝΟΥ ΝΕΩΤΕΡΟΥ ΜΝΗΜΕΙΟΥ ΥΠΟΣΤΕΓΟΥ Γ' ΤΗΣ ΠΟΛΕΜΙΚΗΣ ΑΕΡΟΠΟΡΙΑΣ ΕΝΤΟΣ ΤΟΥ ΜΗΤΡΟΠΟΛΙΤΙΚΟΥ ΠΑΡΚΟΥ ΠΡΑΣΙΝΟΥ ΚΑΙ ΑΝΑΨΥΧΗΣ ΤΟΥ ΜΗΤΡΟΠΟΛΙΤΙΚΟΥ ΠΟΛΟΥ ΕΛΛΗΝΙΚΟΥ- ΑΓ. ΚΟΣΜΑ»
Προέγκριση Οικοδομικής Αδείας (ν.4759/2020): ΕΡΓΑΣΙΕΣ ΑΠΟΚΑΤΑΣΤΑΣΗΣ ΚΕΛΥΦΟΥΣ TOY ΧΑΡΑΚΤΗΡΙΣΜΕΝΟΥ ΝΕΩΤΕΡΟΥ ΜΝΗΜΕΙΟΥ ΥΠΟΣΤΕΓΟΥ Γ' ΤΗΣ ΠΟΛΕΜΙΚΗΣ ΑΕΡΟΠΟΡΙΑΣ ΕΝΤΟΣ ΤΟΥ ΜΗΤΡΟΠΟΛΙΤΙΚΟΥ ΠΑΡΚΟΥ ΠΡΑΣΙΝΟΥ ΚΑΙ ΑΝΑΨΥΧΗΣ ΤΟΥ ΜΗΤΡΟΠΟΛΙΤΙΚΟΥ ΠΟΛΟΥ ΕΛΛΗΝΙΚΟΥ- ΑΓ. ΚΟΣΜΑ»
Έγκριση Εργασιών Δόμησης Μικρής Κλίμακας: ΕΚΤΕΛΕΣΗ ΓΕΩΤΕΧΝΙΚΩΝ ΚΑΙ ΓΕΩΦΥΣΙΚΩΝ ΕΡΕΥΝΩΝ, ΓΕΩΤΕΧΝΙΚΗ ΑΞΙΟΛΟΓΗΣΗ ΚΑΙ ΓΕΩΤΕΧΝΙΚΕΣ ΜΕΛΕΤΕΣ ΘΕΜΕΛΙΩΣΗΣ ΣΤΟ ΧΕΡΣΑΙΟ ΤΜΗΜΑ ΤΟΥ ΜΗΤΡΟΠΟΛΙΤΙΚΟΥ ΠΟΛΟΥ ΕΛΛΗΝΙΚΟΥ –ΑΓ. ΚΟΣΜΑ- ΟΤ ΑΠ 1.5
Έγκριση Εργασιών Δόμησης Μικρής Κλίμακας: Εκτέλεση Γεωτεχνικών και Γεωφυσι

In [9]:
pd.DataFrame(filtered_results).to_csv("oikodomikes_adeies.csv", index=False)

In [10]:
filtered_results = []
for item in results:
    for t in terms:
        if t in item['subject']:
            print(item['subject'])

            filtered_results.append(item)
r = pd.DataFrame(filtered_results)

Προέγκριση Οικοδομικής Αδείας (ν.4759/2020): ΑΝΕΓΕΡΣΗ ΣΥΓΚΡΟΤΗΜΑΤΟΣ 4 ΤΡΙΩΡΟΦΩΝ ΚΤΗΡΙΩΝ ΚΑΤΟΙΚΙΩΝ, ΜΕ ΕΝΑΝ ΥΠΟΓΕΙΟ ΟΡΟΦΟ ΣΤΑΘΜΕΥΣΗΣ ΚΑΙ ΒΟΗΘΗΤΙΚΩΝ ΧΩΡΩΝ, ΦΥΤΕΜΕΝΑ ΔΩΜΑΤΑ, 18 ΚΟΛΥΜΒΗΤΙΚΕΣ ΔΕΞΑΜΕΝΕΣ , ΜΕ ΔΙΑΜΟΡΦΩΣΗ ΠΕΡΙΒΑΛΛΟΝΤΟΣ ΧΩΡΟΥ , ΑΔΕΙΑ ΚΟΠΗΣ ΔΕΝΤΡΩΝ ΚΑΙ ΧΡΗΣΗ ΤΟΥ ΑΡΘΡΟΥ 25 ΤΟΥ ΝΟΚ ΟΠΩΣ ΙΣΧΥΕΙ ΣΤΟ ΟΙΚΟΔΟΜΙΚΟ ΤΕΤΡΑΓΩΝΟ ΠΜ-Π1.6Β
Έγκριση Εργασιών Δόμησης Μικρής Κλίμακας: ΕΚΤΕΛΕΣΗ ΕΡΓΑΣΙΩΝ ΓΙΑ ΓΕΩΤΕΧΝΙΚΗ ΕΡΕΥΝΑ, ΔΟΚΙΜΑΣΤΙΚΕΣ ΤΟΜΕΣ ΕΔΑΦΟΥΣ ΚΑΙ ΕΚΣΚΑΦΕΣ ΣΤΟ ΟΙΚΟΔΟΜΙΚΟ ΤΕΤΡΑΓΩΝΟ Α-Π6.9 (MAINSTREAM II) ΤΟΥ ΜΗΤΡΟΠΟΛΙΤΙΚΟΥ ΠΟΛΟΥ 
ΕΛΛΗΝΙΚΟΥ – ΑΓΙΟΥ ΚΟΣΜΑ 
Έγκριση Εργασιών Δόμησης Μικρής Κλίμακας: ΕΚΤΕΛΕΣΗ ΕΡΓΑΣΙΩΝ ΓΙΑ ΓΕΩΤΕΧΝΙΚΗ ΕΡΕΥΝΑ ΣΤΟ ΟΙΚΟΔΟΜΙΚΟ ΤΕΤΡΑΓΩΝΟ 
 Α-Π6.11 (MAINSTREAM II) ΤΟΥ ΜΗΤΡΟΠΟΛΙΤΙΚΟΥ ΠΟΛΟΥ 
ΕΛΛΗΝΙΚΟΥ – ΑΓΙΟΥ ΚΟΣΜΑ 
Έγκριση Εργασιών Δόμησης Μικρής Κλίμακας: ΕΚΤΕΛΕΣΗ ΕΡΓΑΣΙΩΝ ΓΙΑ ΓΕΩΤΕΧΝΙΚΗ ΕΡΕΥΝΑ, ΔΟΚΙΜΑΣΤΙΚΕΣ ΤΟΜΕΣ ΕΔΑΦΟΥΣ ΚΑΙ ΕΚΣΚΑΦΕΣ ΣΤΟ ΟΙΚΟΔΟΜΙΚΟ ΤΕΤΡΑΓΩΝΟ Α-Π6.10 (MAINSTREAM II) ΤΟΥ ΜΗΤΡΟΠΟΛΙΤΙΚΟΥ ΠΟΛΟΥ 
ΕΛΛΗΝΙΚΟΥ – ΑΓΙΟΥ ΚΟΣΜΑ 
Έγκριση Εργασιών Δόμησης Μικρής Κλίμακ

In [11]:
r.to_csv("oikopeda.csv", index = False)

In [12]:
pd.DataFrame(results).to_csv("ellhniko_all.csv", index=False)

In [13]:
from IPython.display import clear_output

In [14]:
import os
import glob
import pdfplumber

In [15]:
pdfs = glob.glob("/Users/troboukis/Code/lambda/documents/oik_adeies/*")
print(f"Έχουμε {len(pdfs)} pdf")

Έχουμε 153 pdf


In [16]:
save_path = "/Users/troboukis/Code/lambda/documents/oik_adeies/{}.pdf"
docName = [re.search(r"adeies/(.*).pdf", i).group(1) for i in pdfs]
c = 0
for i in filtered_results:
    c+=1
    try:
        if re.search("doc/(.*)", i['documentUrl']).group(1) not in docName:
            response = requests.get(i['documentUrl'])
            print(f"Getting: {c}/{len(filtered_results)}")
            with open(save_path.format(i['ada']), "wb") as f:
                        f.write(response.content)
            with open("download_report.log", "a", encoding="utf-8") as log:
                log.write(f"[{datetime.now().isoformat()}] ADA {i['ada']} downloaded successfully\n")

            time.sleep(1)
            clear_output(wait=True)
    except Exception as e:
        print (e)
        with open("download_report.log", "a", encoding="utf-8") as log:
            log.write(f"[{datetime.now().isoformat()}] ADA {i['ada']} NOT DOWNLOADED\n")
        clear_output(wait=True)

Getting: 157/157


In [17]:
import os
import glob
import pdfplumber
from itertools import chain

pdfs = glob.glob("/Users/troboukis/Code/lambda/documents/oik_adeies/*")
print(f"Έχουμε {len(pdfs)} pdf")

all_parsed = []
counter = 0
total = len(pdfs)
for document in pdfs:
    result = {}
    counter+=1
    ada = os.path.basename(document)
    print(f"{counter}/{total}")
    print(f'parsing: {ada}')
    with pdfplumber.open(document) as pdf:
        parsed = []
        for page in pdf.pages:
            tables = page.extract_tables()
            for table in tables:
                # if any("ΜΕΤΛΕΝ" in (cell or "") for cell in chain.from_iterable(table)):
                #     print(table[0])
                print(table)
                parsed.append(table)
        print("------")
    all_parsed.append(parsed)


Έχουμε 157 pdf
1/157
parsing: 9Γ0Ξ46Ψ842-Γ7Ω.pdf
[['Α/Α Πράξης', '1512911'], ['Ημ/νία έκδοσης πράξης', '09/07/2025'], ['Ισχύει έως', '09/07/2029']]
[['Στοιχεία έργου', None], ['Περιγραφή\nΈργου/Εγκατάστασης', 'ΑΝΕΓΕΡΣΗ ΥΠΟΓΕΙΟΥ ΥΠΟΣΤΑΘΜΟΥ ΜΕΣΗΣ ΤΑΣΗΣ (20kV) ΔΕΔΔΗΕ ΔΙΑΝΟΜΗΣ\nΗΛΕΚΤΡΙΚΗΣ ΕΝΕΡΓΕΙΑΣ στο ΟΤ Α-Π4.10'], ['Οδός', 'O.T. Α-Π4.10 ΠΕΡΙΟΧΗΣ Α-Π4, ΜΗΤΡΟΠΟΛΙΤΙΚΟΣ ΠΟΛΟΣ ΕΛΛΗΝΙΚΟΥ - ΑΓ.ΚΟΣΜΑ'], ['Αρ. από', '-'], ['Αρ. έως', ''], ['Πόλη/Οικισμός', 'ΜΗΤΡΟΠΟΛΙΤΙΚΟΣ ΠΟΛΟΣ ΕΛΛΗΝΙΚΟΥ ΑΓ. ΚΟΣΜΑ'], ['Δήμος', 'Ελληνικού-Αργυρούπολης'], ['Δημοτική Ενότητα /\nΠεριοχή', 'Περιοχή προς Πολεοδόμηση Α-Π4 - Tομέας VΙΙI "Γειτονιά Επιχειρηματικού Κέντρου\nΛεωφόρου Βουλιαγμένης" του Μ.Π. Ελληνικού-Αγ.Κοσμά'], ['ΟΤ', 'Α-Π4.10'], ['ΚΑΕΚ', '050501505002/0/0επιφ'], ['Ηλ. κλειδί πράξης', '7C700C51C441DE6B'], ['Τύπος Πράξης', 'Οικοδομική Άδεια (ν.4759/2020)'], ['A/A Αίτησης', '2014682'], ['Διαχειριστής Αίτησης', 'ΠΟΝΗΡΟΣ ΘΕΟΔΩΡΟΣ (A.M. TEE:138416), ΑΡΧΙΤΕΚΤΟΝΑΣ ΜΗΧΑΝΙΚΟΣ[2015]']]
[['Στοιχεία κυρίου του έργου', N

In [18]:
len(all_parsed)

157

In [19]:
for g in all_parsed[:2]:
    # if "Ιδιοκτήτης" in g[2][2]:
    print(g)

[[['Α/Α Πράξης', '1512911'], ['Ημ/νία έκδοσης πράξης', '09/07/2025'], ['Ισχύει έως', '09/07/2029']], [['Στοιχεία έργου', None], ['Περιγραφή\nΈργου/Εγκατάστασης', 'ΑΝΕΓΕΡΣΗ ΥΠΟΓΕΙΟΥ ΥΠΟΣΤΑΘΜΟΥ ΜΕΣΗΣ ΤΑΣΗΣ (20kV) ΔΕΔΔΗΕ ΔΙΑΝΟΜΗΣ\nΗΛΕΚΤΡΙΚΗΣ ΕΝΕΡΓΕΙΑΣ στο ΟΤ Α-Π4.10'], ['Οδός', 'O.T. Α-Π4.10 ΠΕΡΙΟΧΗΣ Α-Π4, ΜΗΤΡΟΠΟΛΙΤΙΚΟΣ ΠΟΛΟΣ ΕΛΛΗΝΙΚΟΥ - ΑΓ.ΚΟΣΜΑ'], ['Αρ. από', '-'], ['Αρ. έως', ''], ['Πόλη/Οικισμός', 'ΜΗΤΡΟΠΟΛΙΤΙΚΟΣ ΠΟΛΟΣ ΕΛΛΗΝΙΚΟΥ ΑΓ. ΚΟΣΜΑ'], ['Δήμος', 'Ελληνικού-Αργυρούπολης'], ['Δημοτική Ενότητα /\nΠεριοχή', 'Περιοχή προς Πολεοδόμηση Α-Π4 - Tομέας VΙΙI "Γειτονιά Επιχειρηματικού Κέντρου\nΛεωφόρου Βουλιαγμένης" του Μ.Π. Ελληνικού-Αγ.Κοσμά'], ['ΟΤ', 'Α-Π4.10'], ['ΚΑΕΚ', '050501505002/0/0επιφ'], ['Ηλ. κλειδί πράξης', '7C700C51C441DE6B'], ['Τύπος Πράξης', 'Οικοδομική Άδεια (ν.4759/2020)'], ['A/A Αίτησης', '2014682'], ['Διαχειριστής Αίτησης', 'ΠΟΝΗΡΟΣ ΘΕΟΔΩΡΟΣ (A.M. TEE:138416), ΑΡΧΙΤΕΚΤΟΝΑΣ ΜΗΧΑΝΙΚΟΣ[2015]']], [['Στοιχεία κυρίου του έργου', None, None, None, None, None], ['Επώνυμο/ία', '

In [20]:
pd.DataFrame(all_parsed)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,"[[Α/Α Πράξης, 1512911], [Ημ/νία έκδοσης πράξης...","[[Στοιχεία έργου, None], [Περιγραφή\nΈργου/Εγκ...","[[Στοιχεία κυρίου του έργου, None, None, None,...","[[Προέγκριση Οικοδομικής Αδείας (ν.4759/2020),...","[[Εγκρίσεις Φορέων, None, None, None], [Τύπος ...","[[Στοιχεία Διαγράμματος Κάλυψης, None, None, N...","[[ΟΜΑΔΑ ΜΕΛΕΤΗΣ ΕΡΓΟΥ, None], [Μελέτη τοπογραφ...","[[ΟΜΑΔΑ ΕΠΙΒΛΕΨΗΣ ΕΡΓΟΥ, None], [Επίβλεψη Αρχι...","[[ΕΝΤΟΠΙΣΜΟΣ ΑΚΙΝΗΤΟΥ, None], [Συντεταγμένες, ...",None,None,None,None,None,None,None,None,None
1,"[[Α/Α Πράξης, 1143553], [Ημ/νία έκδοσης πράξης...","[[Στοιχεία έργου, None], [Περιγραφή\nΈργου/Εγκ...","[[Στοιχεία κυρίου του έργου, None, None, None,...","[[Πρόσθετες Επιλογές], [Ισχύς Αδείας 4 έτη (Αν...","[[Στοιχεία Διαγράμματος Κάλυψης, None, None, N...","[[Εμβ. κάλυψης κτιρίου, 0, 0, 575,84, 575,84],...","[[ΑΠΑΙΤΟΥΜΕΝΕΣ ΕΓΚΡΙΣΕΙΣ, None], [, Έγκριση Σ....","[[ΟΜΑΔΑ ΜΕΛΕΤΗΣ ΕΡΓΟΥ, None], [Μελέτη τοπογραφ...","[[Ομάδα Ελέγχου Έργου, None], [Τοπογραφικό διά...",None,None,None,None,None,None,None,None,None
2,"[[Α/Α Πράξης, 878905], [Ημ/νία έκδοσης πράξης,...","[[Στοιχεία έργου, None], [Περιγραφή\nΈργου/Εγκ...","[[Στοιχεία κυρίου του έργου, None, None, None,...","[[Πρόσθετες Επιλογές], [Ισχύς Αδείας 6 έτη (Αν...",[[Προέγκριση για Αναθεώρηση Οικ. Αδείας (ν.475...,[[Μεταγενέστερος έλεγχος Οικ. Αδείας (ν.4759/2...,"[[Εγκρίσεις Φορέων, None, None, None], [Τύπος ...","[[Έγκριση Κυκλοφοριακής\nσύνδεσης, ΥΠΟΥΡΓΕΙΟ Υ...","[[Στοιχεία Διαγράμματος Κάλυψης, None, None, N...","[[Χρήσεις ανά όροφο, Ισόγειο, Λοιπές Χρήσεις],...",[[Στοιχεία Δέσμευσης/Εξαγοράς Θέσεων Στάθμευση...,"[[ΟΜΑΔΑ ΜΕΛΕΤΗΣ ΕΡΓΟΥ, None], [Μελέτη τοπογραφ...","[[, ΚΙΡΙΜΛΙΔΗΣ ΔΗΜΗΤΡΙΟΣ (A.M. TEE:15039), ΜΗΧ...","[[ΕΝΤΟΠΙΣΜΟΣ ΑΚΙΝΗΤΟΥ, None], [Συντεταγμένες, ...",None,None,None,None
3,"[[Α/Α Πράξης, 1038346], [Ημ/νία έκδοσης πράξης...","[[Στοιχεία έργου, None], [Περιγραφή\nΈργου/Εγκ...","[[Στοιχεία κυρίου του έργου, None, None, None,...","[[Πρόσθετες Επιλογές], [α. Δοκιμαστικές τομές ...","[[Εγκρίσεις Φορέων, None, None, None], [Τύπος ...","[[ΕΝΤΟΠΙΣΜΟΣ ΑΚΙΝΗΤΟΥ, None], [Συντεταγμένες, ...",None,None,None,None,None,None,None,None,None,None,None,None
4,"[[Α/Α Πράξης, 825961], [Ημ/νία έκδοσης πράξης,...","[[Στοιχεία έργου, None], [Περιγραφή\nΈργου/Εγκ...","[[Στοιχεία κυρίου του έργου, None, None, None,...","[[Πρόσθετες Επιλογές], [Ισχύς αδείας 1 έτος (Ε...","[[Στοιχεία Διαγράμματος Κάλυψης, None], [Εμβαδ...","[[, ΥΦΙΣΤΑΜΕΝΑ, ΝΟΜΙΜΟΠΟΙΟΥ\nΜΕΝΑ, ΠΡΑΓΜΑΤΟΠΟΙ...","[[ΑΠΑΙΤΟΥΜΕΝΕΣ ΕΓΚΡΙΣΕΙΣ, None], [, Έγκριση Σ....","[[ΟΜΑΔΑ ΜΕΛΕΤΗΣ ΕΡΓΟΥ, None], [Μελέτη τοπογραφ...","[[Εφαρμογή χρονικού\nπρογραμματισμού, 314 ARCH...","[[Ομάδα Ελέγχου Έργου, None], [Τοπογραφικό διά...",None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152,"[[Α/Α Πράξης, 1502152], [Ημ/νία έκδοσης πράξης...","[[Στοιχεία έργου, None], [Περιγραφή\nΈργου/Εγκ...","[[Στοιχεία κυρίου του έργου, None, None, None,...","[[Πρόσθετες Επιλογές], [Ισχύς Αδείας 4 έτη (Αν...","[[Στοιχεία Διαγράμματος Κάλυψης, None, None, N...","[[Εμβ. κάλυψης κτιρίου, 0, 0, 0, 0], [Εμβ. δόμ...","[[ΑΠΑΙΤΟΥΜΕΝΕΣ ΕΓΚΡΙΣΕΙΣ, None], [, Έγκριση Σ....","[[ΟΜΑΔΑ ΜΕΛΕΤΗΣ ΕΡΓΟΥ, None], [Μελέτη τοπογραφ...","[[Ομάδα Ελέγχου Έργου, None], [Τοπογραφικό διά...",None,None,None,None,None,None,None,None,None
153,"[[Α/Α Πράξης, 842969], [Ημ/νία έκδοσης πράξης,...","[[Στοιχεία έργου, None], [Περιγραφή\nΈργου/Εγκ...","[[Στοιχεία κυρίου του έργου, None, None, None,...","[[Προέγκριση Οικοδομικής Αδείας (ν.4759/2020),...","[[Εγκρίσεις Φορέων, None, None, None], [Τύπος ...","[[Στοιχεία Διαγράμματος Κάλυψης, None, None, N...",[[Στοιχεία Δέσμευσης/Εξαγοράς Θέσεων Στάθμευση...,"[[Μελέτη τοπογραφικού\nδιαγράμματος, Σ. ΚΑΤΣΩΛ...","[[Επίβλεψη ηλεκτρικών\nεγκαταστάσεων, ΚΑΡΑΓΙΑΝ...","[[ΕΝΤΟΠΙΣΜΟΣ ΑΚΙΝΗΤΟΥ, None], [Συντεταγμένες, ...",None,None,None,None,None,None,None,None
154,"[[Α/Α Πράξης, 1178809], [Ημ/νία έκδοσης πράξης...","[[Στοιχεία έργου, None], [Περιγρα

In [21]:
def clean_parsed_to_dataframe(all_parsed, pdfs):
    """
    Cleans the all_parsed data and creates a structured DataFrame.

    Args:
        all_parsed: List of parsed PDF documents (each is a list of tables)
        pdfs: List of PDF file paths (to extract ADA identifiers)

    Returns:
        pandas DataFrame with cleaned building permit data
    """

    def find_value(tables, key):
        """Search for a key in tables and return its value."""
        for table in tables:
            for row in table:
                if row and len(row) >= 2 and row[0]:
                    clean_key = row[0].replace('\n', ' ').strip() if row[0] else ''
                    if key.lower() in clean_key.lower():
                        return row[1].replace('\n', ' ').strip() if row[1] else None
        return None

    def find_coverage_value(tables, key, column_idx=4):
        """Find values in the coverage diagram table (ΣΥΝΟΛΟ column)."""
        for table in tables:
            for row in table:
                if row and len(row) >= column_idx + 1 and row[0]:
                    clean_key = row[0].replace('\n', ' ').strip() if row[0] else ''
                    if key.lower() in clean_key.lower():
                        for idx in [column_idx, 3, 2, 1]:
                            if len(row) > idx and row[idx]:
                                return row[idx].replace('\n', ' ').strip() if row[idx] else None
        return None

    def find_owner_info(tables):
        """Extract owner information from the owner table."""
        owners = []
        for table in tables:
            if table and len(table) > 1:
                header = table[0] if table[0] else []
                if any('κυρίου' in str(cell).lower() for cell in header if cell):
                    for row in table[1:]:
                        if row and len(row) >= 4:
                            if row[0] and 'Επώνυμο' not in str(row[0]):
                                name = row[0].replace('\n', ' ').strip() if row[0] else ''
                                role = row[3].replace('\n', ' ').strip() if len(row) > 3 and row[3] else ''
                                if name and 'Ιδιοκτήτης' in role:
                                    owners.append(name)
        return '; '.join(owners) if owners else None

    def find_coordinates(tables):
        """Extract coordinates from the location table."""
        for table in tables:
            for row in table:
                if row and len(row) >= 2 and row[0]:
                    if 'Συντεταγμένες' in str(row[0]):
                        return row[1].strip() if row[1] else None
        return None

    records = []

    for idx, (doc, pdf_path) in enumerate(zip(all_parsed, pdfs)):
        if not doc:
            continue

        ada = os.path.basename(pdf_path).replace('.pdf', '')

        record = {
            'ada': ada,
            'aa_praxis': find_value(doc, 'Α/Α Πράξης'),
            'issue_date': find_value(doc, 'Ημ/νία έκδοσης'),
            'valid_until': find_value(doc, 'Ισχύει έως'),
            'description': find_value(doc, 'Περιγραφή'),
            'address': find_value(doc, 'Οδός'),
            'city': find_value(doc, 'Πόλη/Οικισμός'),
            'municipality': find_value(doc, 'Δήμος'),
            'area': find_value(doc, 'Δημοτική Ενότητα'),
            'ot': find_value(doc, 'ΟΤ'),
            'kaek': find_value(doc, 'ΚΑΕΚ'),
            'permit_type': find_value(doc, 'Τύπος Πράξης'),
            'owner': find_owner_info(doc),
            'plot_area': find_value(doc, 'Εμβαδόν οικοπέδου'),
            'coverage_area': find_coverage_value(doc, 'κάλυψης κτιρίου'),
            'building_area': find_coverage_value(doc, 'δόμησης κτιρίου'),
            'uncovered_area': find_coverage_value(doc, 'ακάλυπτου χώρου'),
            'volume': find_coverage_value(doc, 'Όγκος κτιρίου'),
            'max_height': find_coverage_value(doc, 'Μέγιστο ύψος'),
            'floors': find_coverage_value(doc, 'Αριθμός Ορόφων'),
            'parking_spots': find_coverage_value(doc, 'Θέσεων Στάθμευσης'),
            'coordinates': find_coordinates(doc),
        }

        records.append(record)

    df = pd.DataFrame(records)

    # Convert numeric columns
    numeric_cols = ['plot_area', 'coverage_area', 'building_area', 'uncovered_area',
                    'volume', 'max_height', 'floors', 'parking_spots']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Convert date columns
    date_cols = ['issue_date', 'valid_until']
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format='%d/%m/%Y', errors='coerce')

    return df


# Usage:
# df_permits = clean_parsed_to_dataframe(all_parsed, pdfs)
# df_permits.head()

In [22]:
df_permits = clean_parsed_to_dataframe(all_parsed, pdfs)
df_permits.head()


,ada,aa_praxis,issue_date,valid_until,description,address,city,municipality,area,ot,...,owner,plot_area,coverage_area,building_area,uncovered_area,volume,max_height,floors,parking_spots,coordinates
0,9Γ0Ξ46Ψ842-Γ7Ω,1512911,2025-07-09,2029-07-09,ΑΝΕΓΕΡΣΗ ΥΠΟΓΕΙΟΥ ΥΠΟΣΤΑΘΜΟΥ ΜΕΣΗΣ ΤΑΣΗΣ (20kV...,"O.T. Α-Π4.10 ΠΕΡΙΟΧΗΣ Α-Π4, ΜΗΤΡΟΠΟΛΙΤΙΚΟΣ ΠΟΛ...",ΜΗΤΡΟΠΟΛΙΤΙΚΟΣ ΠΟΛΟΣ ΕΛΛΗΝΙΚΟΥ ΑΓ. ΚΟΣΜΑ,Ελληνικού-Αργυρούπολης,"Περιοχή προς Πολεοδόμηση Α-Π4 - Tομέας VΙΙI ""Γ...","Περιοχή προς Πολεοδόμηση Α-Π4 - Tομέας VΙΙI ""Γ...",...,ΕΛΛΗΝΙΚΟ Μ.Α.Ε.,11106.468,0.000,0.00,0.000,0.000,0.0,0.0,0.0,"477244.65011430014 4194368.597070529,477255.23..."
1,6ΥΩ246Ψ842-16Μ,1143553,2024-08-02,2027-08-02,ΑΝΕΓΕΡΣΗ ΝΕΑΣ ΔΙΩΡΟΦΗΣ ΑΥΤΟΤΕΛΟΥΣ ΚΑΤΟΙΚΙΑΣ ΜΕ...,"O.T.ΠΜ-Π1.2 ΠΕΡΙΟΧΗΣ ΠΜ-Π1, ΜΗΤΡΟΠΟΛΙΤΙΚΟΣ ΠΟΛ...",ΑΛΙΜΟΣ,Αλίμου,"ΠΕΡΙΟΧΗ ΠΜ-Π1, ΜΗΤΡΟΠΟΛΙΤΙΚΟΣ ΠΟΛΟΣ ΕΛΛΗΝΙΚΟΥ ...","ΠΕΡΙΟΧΗ ΠΜ-Π1, ΜΗΤΡΟΠΟΛΙΤΙΚΟΣ ΠΟΛΟΣ ΕΛΛΗΝΙΚΟΥ ...",...,SUNSET COVE M.A.E,2094.011,575.840,732.54,1518.170,3412.110,7.8,2.0,10.0,None
2,ΨΣΖΞ46Ψ842-86Ω,878905,2023-11-30,2024-07-14,COMMERCIAL HUB: Β ΑΝΑΘΕΩΡΗΣΗ ΤΗΣ Α/Α ΠΡΑΞΗΣ 74...,"ΠΕΡΙΟΧΗ ΠΡΟΣ ΠΟΛΕΟΔΟΜHΣΗ Α-Π4,ΤΟΜΕΑΣ VI",ΕΛΛΗΝΙΚΟ,Ελληνικού-Αργυρούπολης,ΜΗΤΡΟΠΟΛΙΤΙΚΟΣ ΠΟΛΟΣ ΕΛΛΗΝΙΚΟΥ - ΑΓ.ΚΟΣΜΑ,ΜΗΤΡΟΠΟΛΙΤΙΚΟΣ ΠΟΛΟΣ ΕΛΛΗΝΙΚΟΥ - ΑΓ.ΚΟΣΜΑ,...,ΕΛΛΗΝΙΚΟ Μ.Α.Ε.; LAMDA VOULIAGMENIS S.M.S.A.,122230.782,72818.325,183824.82,49412.457,1129890.031,19.5,6.0,4104.0,"477410.6764880197 4193678.0331894006,477426.55..."
3,ΡΗΘΞ46Ψ842-ΠΑΑ,1038346,2024-04-25,2025-04-25,ΕΚΤΕΛΕΣΗ ΕΡΓΑΣΙΩΝ ΠΕΝΕΤΡΟΜΕΤΡΗΣΕΩΝ ΓΙΑ ΤΗΝ ΕΚΠ...,ΑΝΩΝΥΜΗ,ΕΛΛΗΝΙΚΟ,Ελληνικού-Αργυρούπολης,None,None,...,ALAN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"475050.5884345101 4194092.371518077,475104.034..."
4,6Φ9Ψ46Ψ842-ΜΞ2,825961,2023-10-11,2026-10-11,"ΑΔΕΙΑ ΕΚΣΚΑΦΩΝ, ΜΕ ΣΚΟΠΟ ΤΗΝ ΑΝΕΓΕΡΣΗ ΝΕΟΥ ΣΥΓ...",ΑΝΩΝΥΜΟΣ (Ο.Τ. Α-Π6.10),ΑΛΙΜΟΣ,Αλίμου,ΜΗΤΡΟΠΟΛΙΤΙΚΟΣ ΠΟΛΟΣ ΕΛΛΗΝΙΚΟΥ-ΑΓΙΟΥ ΚΟΣΜΑ,ΜΗΤΡΟΠΟΛΙΤΙΚΟΣ ΠΟΛΟΣ ΕΛΛΗΝΙΚΟΥ-ΑΓΙΟΥ ΚΟΣΜΑ,...,ΕΛΛΗΝΙΚΟ Μ.Α.Ε,6190.667,0.000,0.00,0.000,0.000,0.0,0.0,0.0,None


In [23]:
df_permits.to_csv("permits_ellhniko.csv", index=False)